<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-13/AI_Modul_13.2_Praktikum_Sistem_Deteksi_Berbasis_Video.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 13.2: Praktikum Sistem Deteksi Berbasis Video dan Tracking (SORT)
**Tujuan Pembelajaran:**
1. Membangun modul pelacak objek jamak (Multi-Object Tracking) berbasis algoritma SORT sederhana.
2. Mengimplementasikan asosiasi penugasan optimal Hungarian menggunakan `scipy.optimize.linear_sum_assignment`.
3. Merancang logika penghitungan lintas garis (*line-crossing counting*) pada simulasi video pergerakan truk sawit.

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)
print("Modul praktikum deteksi dan pelacakan video siap!")

### Bagian 1: Implementasi Kelas Tracklet dan Asosiasi Hungarian

In [ ]:
def iou_boxes(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter / union if union > 0 else 0.0

class SimpleTracklet:
    _id_counter = 1
    def __init__(self, bbox):
        self.id = SimpleTracklet._id_counter
        SimpleTracklet._id_counter += 1
        self.bbox = bbox # [x1, y1, x2, y2]
        self.history = [self.center()]
        self.lost_frames = 0
        
    def center(self):
        return ((self.bbox[0] + self.bbox[2])/2.0, (self.bbox[1] + self.bbox[3])/2.0)
    def get_center(self):
        return self.center()
        
    def update(self, bbox):
        self.bbox = bbox
        self.history.append(self.center())
        self.lost_frames = 0

class MiniSORT:
    def __init__(self, iou_thresh=0.3, max_lost=3):
        self.iou_thresh = iou_thresh
        self.max_lost = max_lost
        self.tracks = []
        
    def step(self, detections):
        for t in self.tracks:
            t.lost_frames += 1
            
        if len(self.tracks) == 0:
            for d in detections:
                self.tracks.append(SimpleTracklet(d))
            return self.tracks
            
        # Matriks biaya 1 - IoU
        cost = np.zeros((len(self.tracks), len(detections)))
        for i, t in enumerate(self.tracks):
            for j, d in enumerate(detections):
                cost[i, j] = 1.0 - iou_boxes(t.bbox, d)
                
        row_ind, col_ind = linear_sum_assignment(cost)
        matched_t, matched_d = set(), set()
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] <= (1.0 - self.iou_thresh):
                self.tracks[r].update(detections[c])
                matched_t.add(r)
                matched_d.add(c)
                
        for j, d in enumerate(detections):
            if j not in matched_d:
                self.tracks.append(SimpleTracklet(d))
                
        self.tracks = [t for t in self.tracks if t.lost_frames <= self.max_lost]
        return [t for t in self.tracks if t.lost_frames == 0]

print("Mesin pelacak MiniSORT berhasil dirakit!")

### Bagian 2: Logika Lintas Garis (Virtual Tripwire)

In [ ]:
class Tripwire:
    def __init__(self, y_threshold=200):
        self.y_threshold = y_threshold
        self.counted_ids = set()
        self.total_count = 0
        
    def update(self, tracks):
        for t in tracks:
            if t.id in self.counted_ids or len(t.history) < 2:
                continue
            y_prev = t.history[-2][1]
            y_curr = t.history[-1][1]
            # Melintas dari atas (y < 200) ke bawah (y >= 200)
            if y_prev < self.y_threshold <= y_curr:
                self.counted_ids.add(t.id)
                self.total_count += 1
                print(f"[TRIPWIRE EVENT] Truk TBS ID #{t.id} berhasil dihitung! Total: {self.total_count}")

wire = Tripwire(y_threshold=200)
print("Garis batas virtual terpasang pada y = 200!")

### Bagian 3: Simulasi Pelacakan dan Penghitungan Video 15 Frame

In [ ]:
tracker = MiniSORT(iou_thresh=0.3)
frames_history = []

# Simulasi 2 truk bergerak dari atas ke bawah
# Truk 1: x=100, y bergerak dari 100 ke 300
# Truk 2: x=250, y bergerak dari 140 ke 340
for f_idx in range(12):
    truk1_box = np.array([80, 100 + f_idx * 18, 140, 160 + f_idx * 18], dtype=float)
    truk2_box = np.array([230, 140 + f_idx * 18, 290, 200 + f_idx * 18], dtype=float)
    dets = [truk1_box, truk2_box]
    
    active_tracks = tracker.step(dets)
    wire.update(active_tracks)
    frames_history.append([(t.id, t.bbox.copy(), t.get_center()) for t in active_tracks])

print(f"\nHasil Simulasi Video:")
print(f"  Total Truk Berhasil Dihitung: {wire.total_count} unit")
assert wire.total_count == 2, "Penghitungan gagal! Harusnya tepat 2 unit truk."
print("[VALIDASI SUKSES] 2 truk teridentifikasi secara konsisten tanpa ada ID Switch!")

# Visualisasi Trajektori
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 400)
ax.set_ylim(400, 0) # Format citra

# Garis tripwire
ax.axhline(200, color='red', linestyle='--', linewidth=2, label='Virtual Tripwire (y=200)')
ax.text(10, 190, f"LINE COUNTER (Total: {wire.total_count})", color='red', fontweight='bold', fontsize=10)

# Gambar trajektori pelacakan
colors = ['#1E88E5', '#43A047']
for idx, t in enumerate(tracker.tracks):
    hist = np.array(t.history)
    c = colors[idx % len(colors)]
    ax.plot(hist[:, 0], hist[:, 1], '-o', color=c, label=f'Trajektori Truk #{t.id}')
    # BBox posisi akhir
    b = t.bbox
    rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], linewidth=2, edgecolor=c, facecolor='none')
    ax.add_patch(rect)
    ax.text(b[0], b[1]-5, f"Truk #{t.id}", color=c, fontweight='bold', fontsize=9)

plt.title('Simulasi Pelacakan Objek SORT dan Lintas Garis')
plt.legend()
plt.tight_layout()
plt.savefig('praktikum_video_tracking_sort_13_2.png', dpi=200)
plt.close()
print("[OK] Grafik trajektori disimpan sebagai 'praktikum_video_tracking_sort_13_2.png'")